In [1]:
"""
MULTI-STOCK DAILY SIGNAL GENERATOR
===================================
Generate daily signals using stock-specific optimized models

Usage:
    python multi_stock_signal_generator.py

This script:
1. Loads stock-specific models (optimized_models_U.pkl, etc.)
2. Uses each stock's optimal parameters
3. Generates personalized signals for each stock
"""

import sys
import os
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# Setup paths
#PROJECT_PATH = os.path.dirname(os.path.abspath(__file__))
#sys.path.insert(0, PROJECT_PATH)

from optimize_with_model_persistence import PersistentModelBacktester


class MultiStockSignalGenerator:
    """
    Generate signals using stock-specific models
    """
    
    def __init__(self, stock_configs):
        """
        Initialize multi-stock signal generator
        
        Args:
            stock_configs: dict of {ticker: {'data_file': path, 'model_path': path}}
        """
        self.stock_configs = stock_configs
        self.backtesters = {}
        self.optimal_params = {}
        
    def load_models(self):
        """Load stock-specific models for all stocks"""
        
        print("📦 LOADING STOCK-SPECIFIC MODELS")
        print("="*80)
        
        for ticker, config in self.stock_configs.items():
            model_path = config['model_path']
            
            if not os.path.exists(model_path):
                print(f"⚠️  {ticker:6s} - Model not found: {model_path}")
                continue
            
            try:
                # Load backtester for this stock
                backtester = PersistentModelBacktester(model_path=model_path)
                backtester.load_models()
                
                self.backtesters[ticker] = backtester
                self.optimal_params[ticker] = backtester.optimal_params
                
                print(f"✅ {ticker:6s} - Model loaded from {model_path}")
                
            except Exception as e:
                print(f"❌ {ticker:6s} - Failed to load model: {str(e)[:50]}")
        
        print(f"\n✅ Loaded {len(self.backtesters)}/{len(self.stock_configs)} models successfully")
        print("="*80 + "\n")
    
    def get_signal_for_stock(self, ticker):
        """
        Get trading signal for a single stock using its specific model
        """
        
        if ticker not in self.backtesters:
            return {
                'ticker': ticker,
                'status': 'NO_MODEL',
                'signal': 'HOLD',
                'reason': 'Model not loaded'
            }
        
        config = self.stock_configs[ticker]
        csv_file = config['data_file']
        
        if not os.path.exists(csv_file):
            return {
                'ticker': ticker,
                'status': 'NO_DATA',
                'signal': 'HOLD',
                'reason': 'Data file not found'
            }
        
        try:
            # Load data
            df = pd.read_csv(csv_file, index_col=0)
            df.index = pd.to_datetime(df.index)
            df.columns = [col.capitalize() for col in df.columns]
            
            # Get current info
            current_date = df.index[-1]
            current_price = df['Close'].iloc[-1]
            
            # Get stock-specific backtester and parameters
            backtester = self.backtesters[ticker]
            params = self.optimal_params[ticker]
            
            # Make predictions using this stock's models
            dmd_pred = backtester.dmd_system.make_daily_prediction(
                df,
                params['prediction_horizon']
            )
            
            sindy_pred = backtester.sindy_system.make_daily_prediction(
                df,
                params['prediction_horizon']
            )
            
            if not dmd_pred or not sindy_pred:
                return {
                    'ticker': ticker,
                    'status': 'PRED_FAILED',
                    'signal': 'HOLD',
                    'reason': 'Prediction failed'
                }
            
            # Extract predictions
            dmd_change = dmd_pred['expected_change_pct']
            sindy_change = sindy_pred['expected_change_pct']
            avg_change = (dmd_change + sindy_change) / 2
            
            # Extract directions
            dmd_direction = dmd_pred['direction']
            sindy_direction = sindy_pred['direction']
            
            # Check agreement
            dmd_up = 'UP' in dmd_direction
            sindy_up = 'UP' in sindy_direction
            models_agree = dmd_up == sindy_up
            
            # Generate signal based on THIS STOCK'S optimal parameters
            entry_threshold = params['entry_threshold']
            
            signal = 'HOLD'
            reason = ''
            confidence = 'LOW'
            
            if models_agree:
                if avg_change >= entry_threshold:
                    signal = 'BUY'
                    reason = f'Both models predict +{avg_change:.1f}% (threshold: {entry_threshold}%)'
                    
                    # Confidence based on how much above threshold
                    if avg_change >= entry_threshold * 2:
                        confidence = 'HIGH'
                    elif avg_change >= entry_threshold * 1.5:
                        confidence = 'MEDIUM'
                    else:
                        confidence = 'LOW'
                        
                elif avg_change <= -entry_threshold:
                    signal = 'SELL'
                    reason = f'Both models predict {avg_change:.1f}% (threshold: -{entry_threshold}%)'
                    confidence = 'MEDIUM'
                else:
                    signal = 'HOLD'
                    reason = f'Predicted move ({avg_change:+.1f}%) below threshold (±{entry_threshold}%)'
                    confidence = 'N/A'
            else:
                signal = 'HOLD'
                reason = f'Models disagree: DMD {dmd_change:+.1f}% vs SINDy {sindy_change:+.1f}%'
                confidence = 'N/A'
            
            return {
                'ticker': ticker,
                'status': 'SUCCESS',
                'date': current_date.strftime('%Y-%m-%d'),
                'price': current_price,
                'signal': signal,
                'confidence': confidence,
                'avg_predicted_change': avg_change,
                'dmd_prediction': dmd_change,
                'sindy_prediction': sindy_change,
                'models_agree': models_agree,
                'reason': reason,
                'horizon_days': params['prediction_horizon'],
                'entry_threshold': params['entry_threshold'],
                'take_profit': params['take_profit_pct'],
                'stop_loss': params['stop_loss_pct'],
                'max_holding_days': params['max_holding_days'],
                'model_used': config['model_path']
            }
            
        except Exception as e:
            return {
                'ticker': ticker,
                'status': 'ERROR',
                'signal': 'HOLD',
                'reason': f'Error: {str(e)[:100]}'
            }
    
    def generate_all_signals(self):
        """
        Generate signals for all stocks
        """
        
        print("🎯 GENERATING DAILY SIGNALS (STOCK-SPECIFIC MODELS)")
        print("="*80)
        print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Stocks: {list(self.stock_configs.keys())}")
        print("="*80)
        
        signals = []
        
        for ticker in self.stock_configs.keys():
            print(f"\n📊 Analyzing {ticker}...")
            
            signal = self.get_signal_for_stock(ticker)
            signals.append(signal)
            
            # Print result
            if signal['status'] == 'SUCCESS':
                icon = "🟢" if signal['signal'] == 'BUY' else "🔴" if signal['signal'] == 'SELL' else "⚪"
                print(f"   {icon} {signal['signal']}: {signal['reason']}")
                print(f"      Price: ${signal['price']:.2f} | Predicted: {signal['avg_predicted_change']:+.1f}%")
                print(f"      Model: {signal['model_used']}")
                if signal['signal'] == 'BUY':
                    print(f"      TP: +{signal['take_profit']:.1f}% | SL: {signal['stop_loss']:.1f}% | "
                          f"Max Hold: {signal['max_holding_days']} days")
            else:
                print(f"   ❌ {signal['status']}: {signal['reason']}")
        
        # Create DataFrame
        df_signals = pd.DataFrame(signals)
        
        # Display summary
        self._display_summary(df_signals)
        
        return df_signals
    
    def _display_summary(self, df_signals):
        """Display summary of signals"""
        
        print("\n" + "="*80)
        print("📋 SIGNAL SUMMARY (STOCK-SPECIFIC MODELS)")
        print("="*80)
        
        # Count signals
        buy_signals = df_signals[df_signals['signal'] == 'BUY']
        sell_signals = df_signals[df_signals['signal'] == 'SELL']
        hold_signals = df_signals[df_signals['signal'] == 'HOLD']
        
        print(f"\n🟢 BUY Signals:  {len(buy_signals)}")
        print(f"🔴 SELL Signals: {len(sell_signals)}")
        print(f"⚪ HOLD Signals: {len(hold_signals)}")
        
        # Show BUY signals with stock-specific parameters
        if len(buy_signals) > 0:
            print(f"\n🟢 STOCKS TO BUY (Using Stock-Specific Parameters):")
            print("-"*100)
            for _, row in buy_signals.iterrows():
                if row['status'] == 'SUCCESS':
                    print(f"  {row['ticker']:6s} | ${row['price']:7.2f} | "
                          f"Pred: {row['avg_predicted_change']:+5.1f}% | "
                          f"Conf: {row['confidence']:6s} | "
                          f"Entry>{row['entry_threshold']:.1f}% | "
                          f"TP: +{row['take_profit']:.1f}% | "
                          f"SL: {row['stop_loss']:.1f}%")
        
        # Show SELL signals
        if len(sell_signals) > 0:
            print(f"\n🔴 STOCKS TO SELL:")
            print("-"*80)
            for _, row in sell_signals.iterrows():
                if row['status'] == 'SUCCESS':
                    print(f"  {row['ticker']:6s} | ${row['price']:7.2f} | "
                          f"Pred: {row['avg_predicted_change']:+5.1f}%")
        
        print("\n" + "="*80)
    
    def save_signals(self, df_signals, output_folder='signals'):
        """Save signals to CSV with timestamp"""
        
        # Create output folder
        Path(output_folder).mkdir(exist_ok=True)
        
        # Save timestamped version
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"{output_folder}/signals_multistock_{timestamp}.csv"
        df_signals.to_csv(filename, index=False)
        print(f"\n💾 Signals saved to: {filename}")
        
        # Save latest version
        latest_file = f"{output_folder}/latest_signals.csv"
        df_signals.to_csv(latest_file, index=False)
        print(f"💾 Latest signals: {latest_file}")
        
        return filename


# =============================================================================
# CONFIGURATION
# =============================================================================

def create_stock_configs():
    """
    Configure your stocks and their model paths
    
    IMPORTANT: Each stock must have:
    - data_file: Path to daily CSV
    - model_path: Path to its optimized model (optimized_models_TICKER.pkl)
    """
    
    configs = {
        'U': {
            'data_file': '112025/U_daily.csv',
            'model_path': 'optimized_models_U.pkl'
        },
        'AVGO': {
            'data_file': '112025/AVGO_daily.csv',
            'model_path': 'optimized_models_AVGO.pkl'
        },
        'IONQ': {
            'data_file': '112025/IONQ_daily.csv',
            'model_path': 'optimized_models_IONQ.pkl'
        }
    }
    
    return configs


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def run_multi_stock_signals():
    """
    Main function to generate signals using stock-specific models
    """
    
    print("\n" + "="*80)
    print("🚀 MULTI-STOCK SIGNAL GENERATOR")
    print("   (Using Stock-Specific Optimized Models)")
    print("="*80)
    
    # Get configurations
    stock_configs = create_stock_configs()
    
    # Initialize generator
    generator = MultiStockSignalGenerator(stock_configs)
    
    # Load all models
    generator.load_models()
    
    if len(generator.backtesters) == 0:
        print("\n❌ No models loaded! Run multi_stock_optimizer.py first.")
        return None
    
    # Generate signals
    signals = generator.generate_all_signals()
    
    # Save results
    generator.save_signals(signals)
    
    print("\n✅ Multi-stock signal generation complete!")
    print("="*80)
    
    return signals


if __name__ == "__main__":
    """
    Run daily signal generation for all configured stocks
    """
    
    signals = run_multi_stock_signals()
    
    # Optional: Display clean summary
    if signals is not None:
        print("\n" + "="*80)
        print("📊 TODAY'S TRADING SIGNALS")
        print("="*80)
        
        buy_signals = signals[signals['signal'] == 'BUY']
        
        if len(buy_signals) > 0:
            print("\n🟢 BUY THESE STOCKS:")
            for _, row in buy_signals.iterrows():
                if row['status'] == 'SUCCESS':
                    print(f"\n  📈 {row['ticker']} @ ${row['price']:.2f}")
                    print(f"     Expected: {row['avg_predicted_change']:+.1f}% over {row['horizon_days']} days")
                    print(f"     Take Profit: +{row['take_profit']:.1f}%")
                    print(f"     Stop Loss: {row['stop_loss']:.1f}%")
                    print(f"     Max Hold: {row['max_holding_days']} days")
                    print(f"     Confidence: {row['confidence']}")
                    print(f"     Entry Threshold: >{row['entry_threshold']:.1f}%")
        else:
            print("\n⚪ No BUY signals today")
        
        print("\n" + "="*80)



🚀 MULTI-STOCK SIGNAL GENERATOR
   (Using Stock-Specific Optimized Models)
📦 LOADING STOCK-SPECIFIC MODELS
📦 Loading pre-trained models from optimized_models_U.pkl
✅ Models loaded successfully!

🔬 Training Info:
   Models trained for: 10-day prediction horizon

📊 Optimal Parameters:
   prediction_horizon: 10
   entry_threshold: 0.3
   take_profit_pct: 3.0
   stop_loss_pct: -3.5
   max_holding_days: 9
   profit_window: 6

🎯 Expected Performance (from optimization):
   Total Return: 200.52%
   Sharpe Ratio: 2.35
   Max Drawdown: -15.02%
   Win Rate: 69.77%
   Trades: 43
✅ U      - Model loaded from optimized_models_U.pkl
📦 Loading pre-trained models from optimized_models_AVGO.pkl
✅ Models loaded successfully!

🔬 Training Info:
   Models trained for: 10-day prediction horizon

📊 Optimal Parameters:
   prediction_horizon: 10
   entry_threshold: 3.0
   take_profit_pct: 7.0
   stop_loss_pct: -4.5
   max_holding_days: 9
   profit_window: 6

🎯 Expected Performance (from optimization):
   Total